In [ ]:
import datetime
import glob

import cmocean.cm as cmo
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import xarray as xr

from scipy import interpolate

import parcels

In [ ]:
files = sorted(glob.glob("Loggerhead_Simulation_C*.parquet"))

beach_date = datetime.datetime(2024, 4, 24)

ds_fld = xr.open_dataset("copernicus_physics.nc")
land_mask_condition = (ds_fld.U.data[0, 0, :, :] == 0) & \
                      (ds_fld.V.data[0, 0, :, :] == 0)
landmask = np.ma.masked_where(land_mask_condition, land_mask_condition)
lon, lat = ds_fld.lon, ds_fld.lat
lon_centers = lon[1:] - np.diff(lon) / 2
lat_centers = lat[1:] - np.diff(lat) / 2
x, y = np.meshgrid(lon_centers, lat_centers)

fl = interpolate.RectBivariateSpline(lat, lon, landmask.mask, kx=1, ky=1)
lmask = (fl(y[:, 0], x[0, :]) > 0.95).astype(int)
ds_fld.close()

for file in files:
    df = parcels.read_particlefile(file)
    df = df.with_columns(
        pl.col("temperature").replace(0, np.nan).alias("temperature")
    )

    fig = plt.figure(figsize = (10, 5))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_facecolor('aliceblue')
    ax.gridlines(draw_labels=['left','bottom'], zorder=2, alpha=0.3, linestyle='--')

    ax.pcolormesh(x, y, lmask, cmap='Greys', alpha=0.5, zorder=0)

    for traj in df.partition_by("particle_id", maintain_order=True):
        plt.plot(traj["x"], traj["y"], color="gray", alpha=0.5, zorder=1)
        scat = ax.scatter(
            traj["x"],
            traj["y"],
            c=traj["temperature"],
            s=3,
            zorder=3,
            cmap=cmo.thermal,
            vmin=5,
            vmax=20,)
        ax.plot(traj["x"][0], traj["y"][0], "co", zorder=4)
        ax.plot(traj["x"][-1], traj["y"][-1], "ko", zorder=4)

    ax.set_xlim([-22, 10])
    ax.set_ylim([43, 58])
    plt.xlabel('Longitude')
    plt.colorbar(scat, ax=ax, label='Sea surface temperature (°C)')
    plt.ylabel('Latitude')
    runtime = (df["t"].max() - df["t"].min()).days
    ID = file.split('_')[-1].split('.')[0]
    beach_date = df['t'].max().strftime('%Y-%m-%d')
    plt.title(f"{runtime}-day Loggerhead particle trajectories for {ID} (found {beach_date})")
    plt.savefig(file.replace('.parquet', '.png'), dpi=300, bbox_inches='tight')
    plt.show()